In [2]:
import sys
import os
from dataclasses import asdict, dataclass
import numpy as np
import pandas as pd
from category_encoders import TargetEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

# ==========================================
# 1. CONFIGURACIÓN Y CARGA DE DATOS
# ==========================================
ruta_raiz = r"C:\Users\marli\Downloads\vehicle-emissions-classification-main (1)\vehicle-emissions-classification-main"
if ruta_raiz not in sys.path:
    sys.path.append(ruta_raiz)

from src.preprocessing import LowDoc, franchise_code, urban_rural, RevLineCr, newExists
from src.preprocessing import base_cleaning

print("Iniciando tubería de preprocesamiento con codificación por objetivo y características multiplicativas...")

ruta_train = os.path.join(ruta_raiz, "data", "train.csv")
ruta_test = os.path.join(ruta_raiz, "data", "test_nolabel.csv")

df_train = pd.read_csv(ruta_train)
df_test = pd.read_csv(ruta_test)
ids_test = df_test['id'].copy()

# ==========================================
# 2. DEFINICIÓN DE OPCIONES Y TUBERÍA PRINCIPAL
# ==========================================
@dataclass
class OneStepOptions:
    # Se conservan las opciones originales para preservar la integridad de los datos
    noemp_option: str = "raw"  
    newexist_option: str = "A"
    franchise_option: str = "binary"
    urbanrural_option: str = "onehot"
    revlinecr_option: str = "C"
    lowdoc_option: str = "C"
    local_state: str = "IL"

# El codificador por objetivo se instancia fuera de la función para garantizar
# que el ajuste sobre entrenamiento sea reutilizado correctamente en la fase de prueba
codificador_te = TargetEncoder(cols=["City", "Bank"], smoothing=0.3)

def preprocess_one_step_grandmaster(df: pd.DataFrame, te: TargetEncoder, is_train: bool, options: OneStepOptions = OneStepOptions()) -> pd.DataFrame:
    df_out = df.copy()

    # 1. Limpieza base de columnas
    df_out = base_cleaning.clean_base_columns(df_out, local_state=options.local_state)

    # 2. Recuperación de valores numéricos en bruto para preservar la precisión en modelos de árbol
    if 'DisbursementGross' in df_out.columns:
        df_out['Disbursement_Raw'] = pd.to_numeric(df_out['DisbursementGross'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False), errors='coerce').fillna(0)
    for col in ['CreateJob', 'RetainedJob', 'NoEmp']:
        if col in df_out.columns:
            df_out[col + '_Raw'] = pd.to_numeric(df_out[col], errors='coerce').fillna(0)

    # 3. Codificación por objetivo (Target Encoding) aplicada de forma diferenciada
    #    según la partición: ajuste y transformación en entrenamiento,
    #    solo transformación en prueba
    if "City" in df_out.columns and "Bank" in df_out.columns:
        if is_train and "Accept" in df_out.columns:
            # Partición de entrenamiento: ajuste del codificador y transformación simultánea
            df_out[["City_TE", "Bank_TE"]] = te.fit_transform(df_out[["City", "Bank"]], df_out["Accept"])
        else:
            # Partición de prueba: transformación usando los parámetros aprendidos en entrenamiento
            df_out[["City_TE", "Bank_TE"]] = te.transform(df_out[["City", "Bank"]])
    
    # 4. Construcción de variable temporal: antigüedad del préstamo en días
    if "ApprovalDate" in df_out.columns:
        fechas = pd.to_datetime(df_out["ApprovalDate"], format='%d-%b-%y', errors='coerce')
        df_out["LoanAgeDays"] = (pd.Timestamp("2020-01-01") - fechas).dt.days
        df_out["LoanAgeDays"] = df_out["LoanAgeDays"].fillna(df_out["LoanAgeDays"].median())

    # 5. Aplicación de módulos de preprocesamiento específicos por variable
    df_out = LowDoc.preprocess_lowdoc(df_out, option=options.lowdoc_option)
    df_out = RevLineCr.preprocess_revlinecr(df_out, option=options.revlinecr_option)
    df_out = franchise_code.preprocess_franchise_code(df_out, option=options.franchise_option)
    df_out = urban_rural.preprocess_urban_rural(df_out, option=options.urbanrural_option)
    df_out = newExists.preprocess_newexist(df_out, option=options.newexist_option)

    # ============================================================
    # FASE 2 — INGENIERÍA DE CARACTERÍSTICAS AVANZADA
    # Se construyen características multiplicativas para capturar
    # interacciones no lineales relevantes para modelos de árbol
    # ============================================================

    # Interacción entre antigüedad del préstamo y monto desembolsado
    if "LoanAgeDays" in df_out.columns and "Disbursement_Raw" in df_out.columns:
        df_out["LoanAge_x_Disbursement"] = df_out["LoanAgeDays"] * df_out["Disbursement_Raw"]

    # Interacción entre la codificación de ciudad y el número de empleados
    if "City_TE" in df_out.columns and "NoEmp_Raw" in df_out.columns:
        df_out["City_TE_x_NoEmp"] = df_out["City_TE"] * df_out["NoEmp_Raw"]

    # Interacción entre la codificación del banco y la antigüedad del préstamo
    if "Bank_TE" in df_out.columns and "LoanAgeDays" in df_out.columns:
        df_out["Bank_TE_x_LoanAge"] = df_out["Bank_TE"] * df_out["LoanAgeDays"]

    # Consolidación de indicadores de línea de crédito renovable
    if "has_RevLineCr" in df_out.columns and "revlinecr_is_nonstandard" in df_out.columns:
        df_out["RevLineCr_flag"] = (df_out["has_RevLineCr"] | df_out["revlinecr_is_nonstandard"]).astype(int)

    # Consolidación de indicadores del programa de documentación reducida
    if "is_LowDoc" in df_out.columns and "lowdoc_is_nonstandard" in df_out.columns:
        df_out["LowDoc_flag"] = (df_out["is_LowDoc"] | df_out["lowdoc_is_nonstandard"]).astype(int)

    # 6. Eliminación de columnas no informativas o redundantes
    cols_to_drop = [
        "Zone_Undefined", "lowdoc_is_missing", "newexist_missing_or_invalid", "lowdoc_is_nonstandard",
        "City", "Bank", "State", "Zip", "BankState", "ApprovalDate", "ApprovalFY", 
        "DisbursementDate", "Name", "LoanNr_ChkDgt", "BalanceGross", "DisbursementGross",
        "CreateJob", "RetainedJob", "NoEmp"
    ]
    df_out = df_out.drop(columns=[c for c in cols_to_drop if c in df_out.columns], errors='ignore')

    return df_out

# ==========================================
# 3. APLICACIÓN DE LA TUBERÍA Y ALINEACIÓN DE COLUMNAS
# ==========================================
print("Aplicando tubería de preprocesamiento al conjunto de entrenamiento...")
df_train = preprocess_one_step_grandmaster(df_train, te=codificador_te, is_train=True)
df_train['Accept'] = df_train['Accept'].astype(float)
df_train = df_train.dropna(subset=['Accept'])

print("Aplicando tubería de preprocesamiento al conjunto de prueba...")
df_test = preprocess_one_step_grandmaster(df_test, te=codificador_te, is_train=False)

y_completo = df_train["Accept"]
X_completo = df_train.drop(columns=["Accept"]).select_dtypes(include=['number', 'bool']).fillna(0).astype(float)
X_test_final = df_test.select_dtypes(include=['number', 'bool']).fillna(0).astype(float)

# Alineación de columnas entre entrenamiento y prueba para garantizar consistencia
X_completo, X_test_final = X_completo.align(X_test_final, join='left', axis=1, fill_value=0)

# Estandarización de características mediante escalado z-score
scaler = StandardScaler()
X_completo_scaled = scaler.fit_transform(X_completo) 
X_test_final_scaled = scaler.transform(X_test_final)

# ==========================================
# 4. ENTRENAMIENTO DEL MODELO (Random Forest)
# ==========================================
print("Entrenando modelo de random forest con configuración optimizada...")

modelo_rf = RandomForestClassifier(
    n_estimators=500,           
    max_depth=12,               
    min_samples_leaf=5,         
    class_weight='balanced',    
    random_state=42,
    n_jobs=-1
)

modelo_rf.fit(X_completo_scaled, y_completo)

print("Generando archivo de predicciones...")
probabilidades = modelo_rf.predict_proba(X_test_final_scaled)[:, 1]
predicciones = (probabilidades >= 0.50).astype(int)

df_entrega = pd.DataFrame({
    "id": ids_test,
    "Accept": predicciones
})

ruta_csv = os.path.join(ruta_raiz, "submission_bagging_random_forest.csv")
df_entrega.to_csv(ruta_csv, index=False)

print(f"Proceso finalizado. Archivo de entrega generado en: {ruta_csv}")


Iniciando tubería de preprocesamiento con codificación por objetivo y características multiplicativas...
Aplicando tubería de preprocesamiento al conjunto de entrenamiento...
Aplicando tubería de preprocesamiento al conjunto de prueba...
Entrenando modelo de random forest con configuración optimizada...
Generando archivo de predicciones...
Proceso finalizado. Archivo de entrega generado en: C:\Users\marli\Downloads\vehicle-emissions-classification-main (1)\vehicle-emissions-classification-main\submission_bagging_random_forest.csv
